# Create File Train/Val/Test Split

In [ ]:
import os 
#change current data directory to the root of the project
os.chdir("/root/birdclef-ml-framework/")

In [ ]:
import pandas as pd 
import numpy as np 
import ast
import matplotlib.pyplot as plt
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from src.util.stats import get_label_distribution_count_for_soundscapes, get_label_distribution_count_for_train_audio, get_perch_labels
from more_itertools import powerset

In [ ]:
def get_train_soundscapes_df(path):
    df = pd.read_csv(path)
    print(f"Original shape: {df.shape}")
    
    # Group by the specified columns and aggregate 'primary_label'
    df = df.groupby(['filename', 'start', 'end'], as_index=False).agg({
        'primary_label': lambda x: list(set(';'.join(x.dropna().astype(str)).split(';')))
    })
    #map back to string labels with ; seperator 
    df['primary_label'] = df['primary_label'].apply(lambda x: ';'.join(x))

    df["start"] = pd.to_timedelta(df["start"])
    df["end"] = pd.to_timedelta(df["end"])
    
    print(f"Fused shape: {df.shape}")
    return df

In [ ]:
taxonomy_df = pd.read_csv("./data/birdclef_dataset/taxonomy.csv")
sample_submission_df = pd.read_csv("./data/birdclef_dataset/sample_submission.csv")
train_soundscapes_labels_df = get_train_soundscapes_df("./data/birdclef_dataset/train_soundscapes_labels.csv")


train_soundscapes_labels_df['start'] = pd.to_timedelta(train_soundscapes_labels_df['start'])
train_soundscapes_labels_df['end'] = pd.to_timedelta(train_soundscapes_labels_df['end'])
train_df = pd.read_csv("./data/birdclef_dataset/train.csv")
train_df["unique_file_id"] = train_df["filename"].apply(lambda x: x.split("/")[-1])
perch_labels = get_perch_labels()
lower_quantile = 52 #out of first visualization.ipynb

train_audio_path = "./data/birdclef_dataset/train_audio/"
train_soundscapes_path = "./data/birdclef_dataset/train_soundscapes/"

label_to_scientific_name = dict(zip(taxonomy_df["common_name"], taxonomy_df["scientific_name"]))

In [ ]:
def get_file_names(audio_folder_path):
    file_names = []
    
    #recursively walk through the audio folder and its subfolders 
    for root, dirs, files in os.walk(audio_folder_path):
        for file in files:
            if file.endswith(".ogg"):
                complete_file_path = os.path.join(root, file)
                file_names.append(complete_file_path)
    return file_names

In [ ]:
def get_train_audio_labels(file_path):
    #get the label for the given filename from the train_df
    #get file
    filename = os.path.basename(file_path)
    label = train_df[train_df["unique_file_id"] == filename]["primary_label"].values[0]
    secondary_labels = train_df[train_df["unique_file_id"] == filename]["secondary_labels"].values[0]
    labels = [label]
    
    if pd.notna(secondary_labels):
        #get as list 
        secondary_labels = ast.literal_eval(secondary_labels)
        labels.extend(secondary_labels)  # Add secondary labels to the list of labels
    labels = list(set(labels))  # Remove duplicates if any
    labels = [taxonomy_df[taxonomy_df["primary_label"] == label]["common_name"].values[0] for label in labels]
    return labels 

def get_train_soundscapes_labels(file_path):
    #get the label for the given filename from the train_soundscapes_labels_df
    filename = os.path.basename(file_path)
    labels = []
    print(filename)
    primary_labels_windows = train_soundscapes_labels_df[train_soundscapes_labels_df["filename"] == filename]["primary_label"]
    print("primary_labels_windows:", primary_labels_windows)
    for primary_labels_window in primary_labels_windows:
        window_labels = primary_labels_window.split(";")
        print("window_labels:", window_labels)  # Remove leading/trailing whitespace
        labels.extend(window_labels)
    labels = list(set(labels))  # Remove duplicates if any
    labels = [taxonomy_df[taxonomy_df["primary_label"] == label]["common_name"].values[0] for label in labels]
    return labels

In [ ]:
rerun = False
if rerun:
    #filename; label 
    train_audio_files = get_file_names(train_audio_path)
    train_soundscapes_files = get_file_names(train_soundscapes_path)

    train_audio_dict = {filename: get_train_audio_labels(filename) for filename in train_audio_files}
    train_soundscapes_dict = {filename: get_train_soundscapes_labels(filename) for filename in train_soundscapes_files}

    #fuse the two dictionaries
    fused_dict = {**train_audio_dict, **train_soundscapes_dict}

    #df 
    all_files_df = pd.DataFrame(list(fused_dict.items()), columns=["filename", "primary_label"])
    print("Number of files with labels:", len(all_files_df))
    #filter out rows with empty labels
    all_files_df = all_files_df[all_files_df["primary_label"].map(len) > 0]
    print("Number of files with non-empty labels:", len(all_files_df))

    all_files_df.to_csv("fused_labels.csv", index=False)

else:
    all_files_df = pd.read_csv("fused_labels.csv")
    all_files_df["primary_label"] = all_files_df["primary_label"].apply(ast.literal_eval)

## Extra Rules for Labels below 1.Quantile Count

if below count of 1.quantile, files will be handled differently and the following rules apply in order 

RULES: 

- 1. if all lower_freq labels handled by perch -> test only 
- 2. is sonotype or Guaraní leaf-litter frog (soundscapes only ): 
    - try to get 1/3 train and 2/3 test for slice_count over all files and time_slices  
- 3. has unique family in taxonomy: -> test only  (marmoset,caiman,Hooded Capuchin) if possible
- 4. special case 2 froggys 
    -> Guaraní leaf-litter frog(count 8) -> like sonotype 
    -> Uruguay Harlequin Frog (count 5) -> test only 


Important:
- (after each rule apply: check whether new cases popped up in this rule (rerun vs assert) 
- (remove of file might result in other to have one file less(rerun)  or even worse if it is not even there anymore(assert)))
- (check needs to be done for special cases files and also for the rest(no new below quantile cases ))

In [ ]:
special_cases_train = []
special_cases_test = []

In [ ]:
# lambda function to get_rule 
is_below_lower_quantile = lambda label_name, label_file_count: label_file_count[label_name] < lower_quantile
is_only_one_file = lambda label_name, label_file_count: label_file_count[label_name] == 1
is_low_frequency_soundscapes = lambda label_name: "sonotype" in label_name.lower()  or label_name in["Guaraní leaf-litter frog", "Chiasmocleis mehelyi"]
is_test_only_label = lambda label_name: label_name.strip() in ["Southern Spectacled Caiman", "Hooded Capuchin", "Black-tailed Marmoset", "Uruguay Harlequin Frog"]
is_perch_only = lambda scientific_names: all([scientific_name.strip().lower() in perch_labels for scientific_name in scientific_names])

In [ ]:
#split all_files_df into lower_quantile_df and above_quantile_df based on the label distribution count
def get_label_file_count():
    label_file_count = get_label_distribution_count_for_soundscapes(train_soundscapes_labels_df, taxonomy_df)
    label_file_count.update(get_label_distribution_count_for_train_audio(train_df, taxonomy_df))
    return label_file_count
                         
def get_below_lower_quantile_df(all_files_df):
    label_file_count = get_label_file_count()
    all_files_df["below_lower_quantile"] = all_files_df["primary_label"].apply(lambda label_list: any(is_below_lower_quantile(label, label_file_count) for label in label_list))
    return all_files_df[all_files_df["below_lower_quantile"] == True]

#split all_files_df into lower_quantile_df and above_quantile_df based on the rules via index
lower_quantile_df = get_below_lower_quantile_df(all_files_df)
above_quantile_df = all_files_df[~all_files_df.index.isin(lower_quantile_df.index)]
label_file_count = get_label_file_count()

only_one_file_labels = [label for label, count in label_file_count.items() if count == 1]
print("Labels with only one file:", only_one_file_labels)

assert len(lower_quantile_df) + len(above_quantile_df) == len(all_files_df)




#### Perch Rule

In [ ]:
print("Before Perch Rule", len(lower_quantile_df))
for lower_quantile_df_index, row in lower_quantile_df.copy(deep=True).iterrows():
    label_list = row["primary_label"]
    #map to scientific name 
    scientific_names = [label_to_scientific_name.get(label, label) for label in label_list]
    if is_perch_only(scientific_names):
            special_cases_test.append((row["filename"], row["primary_label"]))
            for label in label_list:
                label_file_count[label] -= 1  # Decrement the count for this label
            lower_quantile_df = lower_quantile_df.drop(lower_quantile_df_index)  # Remove this file from lower_quantile_df
print("After Perch Rule", len(lower_quantile_df))

#### Catch is_test_only that only have their own files 

In [ ]:
lonely_labels = list(set([row["primary_label"][0] for _, row in lower_quantile_df.iterrows() if len(row["primary_label"]) == 1]))

for lonely_label in lonely_labels:
    #check if the label has only its own files in the lower_quantile_df
    for lower_quantile_df_index, row in lower_quantile_df.copy(deep=True).iterrows():
        if lonely_label in row["primary_label"]:
            #check if the file has only this label
            if len(row["primary_label"]) == 1 and (is_test_only_label(lonely_label)):
                special_cases_test.append((row["filename"], row["primary_label"]))
                label_file_count[lonely_label] -= 1  # Decrement the count for this label
                lower_quantile_df = lower_quantile_df.drop(lower_quantile_df_index)  # Remove this file from lower_quantile_df
print("After Lonely Label Rule", len(lower_quantile_df))

### Check that from now on only soundscapes are handled

In [ ]:
#assert that all other files are soundscape files 
for lower_quantile_df_index, row in lower_quantile_df.iterrows():
    filename = row["filename"]
    if "train_soundscapes" not in filename:
        raise ValueError(f"File {filename} is not a soundscape file but is in the lower_quantile_df.")

### Catch low frequency without sonotypes 


In [ ]:

for lower_quantile_df_index, row in lower_quantile_df.copy(deep=True).iterrows():
    label_list = row["primary_label"]
    
    has_label_of_sonotype = [label for label in label_list if "sonotype" in label.lower()]
    if has_label_of_sonotype:
        continue

    for label in label_list:
        if is_test_only_label(label):
            special_cases_test.append((row["filename"], row["primary_label"]))
            for label in label_list:
                print(label)
                assert not "sonotype" in label.lower()  # Ensure that low frequency sonotypes are not included in this rule
                label_file_count[label] -= 1  # Decrement the count for this label
            lower_quantile_df = lower_quantile_df.drop(lower_quantile_df_index)  # Remove this file from lower_quantile_df
            continue

print("After other is_test_only_label", len(lower_quantile_df))

### Check if is_low_frequency_soundscapes

In [ ]:
#assert that all other files are soundscape files 
for lower_quantile_df_index, row in lower_quantile_df.iterrows():
    filename = row["filename"]
    if not any(is_low_frequency_soundscapes(label) for label in row["primary_label"]):
        raise ValueError(f"File {filename} does not contain low frequency soundscape labels but is in the lower_quantile_df.")
        

In [ ]:
# to get time stamp specific label distribution 
def get_label_time_stamp_count_dict(df,start="00:00:00", end="00:01:00",key_filter_func=None): 
    start = pd.to_timedelta(start)
    end = pd.to_timedelta(end)
    
    label_time_stamp_count_dict = {}
    
    for _, row in df.iterrows():
        file_name = row["filename"].split("/")[-1]
        file_time_stamps_df = train_soundscapes_labels_df[train_soundscapes_labels_df["filename"] == file_name]
        for time_stamp_row in file_time_stamps_df.itertuples():
            time_stamp_primary_labels = time_stamp_row.primary_label.split(";")
            if not (time_stamp_row.start >= start and time_stamp_row.end <= end):
                continue

            for label in time_stamp_primary_labels:
                if label not in label_time_stamp_count_dict:
                    label_time_stamp_count_dict[label] = 0
                if label in time_stamp_primary_labels:
                    label_time_stamp_count_dict[label] += 1
    
    #map keys back to common names
    label_time_stamp_count_dict = {taxonomy_df[taxonomy_df["primary_label"] == label]["common_name"].values[0]: count for label, count in label_time_stamp_count_dict.items()}
    
    # filter keys based on key_filter_func if provided
    if key_filter_func is not None:
        label_time_stamp_count_dict = {label: count for label, count in label_time_stamp_count_dict.items() if key_filter_func(label)}
    return label_time_stamp_count_dict

### Try out simple slicing (did not work)

In [ ]:
# showcase simple slicing for every file 
# best case for split 1/3 train and 2/3 test 
potential_time_slices = [("00:00:00", f"00:00:{str(i).zfill(2)}") for i in range(5,60,5)]
for _, end in potential_time_slices:
    print(f"\nTime slice: {end}")
    print("Train: ", get_label_time_stamp_count_dict(lower_quantile_df, end=end, key_filter_func = is_low_frequency_soundscapes))
    print("Test: ", get_label_time_stamp_count_dict(lower_quantile_df, start=end, key_filter_func = is_low_frequency_soundscapes))

### Prepare Data for Low Frequency Sonotypes

In [ ]:
label_time_stamp_count_dict = get_label_time_stamp_count_dict(lower_quantile_df)
#remove labels that are not is_low_frequency_soundscapes
label_time_stamp_count_dict = {label: count for label, count in label_time_stamp_count_dict.items() if is_low_frequency_soundscapes(label)}
print("Label time stamp count before splitting:", label_time_stamp_count_dict)

max_key = max([count for label, count in label_time_stamp_count_dict.items()]) +1 
#sort lower_quantile_df by the sum of label_time_stamp_count_dict values for each file
lower_quantile_df["urgency"] = lower_quantile_df["primary_label"].apply(lambda label_list: min(label_time_stamp_count_dict.get(label, max_key) if is_low_frequency_soundscapes(label) else max_key for label in label_list))

lower_quantile_df = lower_quantile_df.sort_values(by="urgency", ascending=True)

# to showcase distributions later on 
train_label_time_stamp_count = dict.fromkeys(label_time_stamp_count_dict.keys(), 0)
test_label_time_stamp_count = dict.fromkeys(label_time_stamp_count_dict.keys(), 0)
    

In [ ]:
#precompute soundscapes_time_stamps_positions
def get_soundscapes_time_stamps_positions():
    time_stamp_starts = [f"00:00:{str(i).zfill(2)}" for i in range(0,60,5)]
    soundscapes_time_stamps_positions = {label: {} for label, count in label_time_stamp_count_dict.items()} # label: {file: slice_point}
    for label, count in sorted(label_time_stamp_count_dict.items(),key=lambda item: item[1]):

        files = lower_quantile_df[lower_quantile_df["primary_label"].apply(lambda label_list: label in label_list)]["filename"].tolist() 
        primary_label = taxonomy_df[taxonomy_df["common_name"] == label]["primary_label"].values[0]

        for file in files:
            soundscapes_time_stamps_positions[label][file] = []

        for file in files:
            file_time_stamps_df = train_soundscapes_labels_df[train_soundscapes_labels_df["filename"] == file.split("/")[-1]]
            count_for_file = 0
            for slice_idx, start in enumerate(time_stamp_starts):
                is_in_slice = file_time_stamps_df[(file_time_stamps_df["start"] == start)].apply(lambda row: primary_label in row["primary_label"].split(";"), axis=1).any()
                if bool(is_in_slice): 
                    soundscapes_time_stamps_positions[label][file].append(slice_idx)



    return soundscapes_time_stamps_positions


def transpose_dict_to_file(soundscapes_time_stamps_positions):
    file_time_stamps_positions = {}
    for label, file_positions in soundscapes_time_stamps_positions.items():
        has_one_file_only = len(list(file_positions.keys())) == 1
        for file, positions in file_positions.items():
            if file not in file_time_stamps_positions:
                file_time_stamps_positions[file] = {"labels": {}, "only_one": has_one_file_only}
            file_time_stamps_positions[file]["labels"][label] = positions
            if "only_one" not in file_time_stamps_positions[file]:  # Only set "only_one" if it hasn't been set before
                file_time_stamps_positions[file]["only_one"] = has_one_file_only  
            else: 
                file_time_stamps_positions[file]["only_one"] = file_time_stamps_positions[file]["only_one"] or has_one_file_only  # Update "only_one" to be True only if all labels for this file have one file only
    return file_time_stamps_positions


### Sonotypes Split (with cost function to minimize)

In [ ]:
def calculate_split_cost(train_dist, test_dist, train_ratio, label_counts, label_weights, penalty_weight=2):

    desired_labels_test_ratio = {label: 1-train_ratio for label in label_counts.keys()}

    total_cost = 0
    for label in label_counts.keys():
        #calculate distance from desired ratio for test distribution 
        test_distance = ( test_dist[label] - desired_labels_test_ratio[label]) ** 2 

        #calculate distance from desired ratio for train distribution 
        train_distance = (train_dist[label] - train_ratio) ** 2

        total_cost += test_distance + train_distance 
        label_weight = label_weights.get(label, 1/len(label_counts))  # Default weight is inversely proportional to the number of labels

        if train_dist[label] == 0 or test_dist[label] == 0:
            total_cost *= penalty_weight * label_weight  # Apply penalty if one of the distributions is zero

        total_cost *= label_weight
        
    return total_cost

def get_max_label_cost(train_dist, test_dist, train_ratio, label_counts, label_weights):
    costs = []
    for label in label_counts.keys():
        label_train = {label: train_dist[label]}
        label_test = {label: test_dist[label]}
        label_count = {label: label_counts[label]}
        label_weight = {label: label_weights.get(label, 1)}
        cost = calculate_split_cost(label_train, label_test, train_ratio, label_count, label_weight)
        costs.append(cost)
    return max(costs)

def get_label_weights(label_time_stamp_count_dict, only_first_few=False):
    complete_counts = sum(label_time_stamp_count_dict.values())
    label_weights = {label: complete_counts / count if count > 0 else 0 for label, count in label_time_stamp_count_dict.items()}
    
    if only_first_few: 
        sorted_labels = sorted(label_time_stamp_count_dict.items(), key=lambda item: item[1])
        for idx, (label, count) in enumerate(sorted_labels):
            if idx < 3:  # Give more weight to the first few labels
                label_weights[label] 
            else:
                label_weights[label]= 1/complete_counts

    return label_weights

def find_optimal_time_slice_for_labels(file, labels): 

    labels_distribution_count = {label: len(positions) for label, positions in per_file_time_stamps_positions_dict[file]["labels"].items() if label in labels}
    slices = []
    for idx, train_time_slice in enumerate(potential_time_slices):
        
        train_distribution = {label: 0 for label in labels_distribution_count.keys()}
        test_distribution = {label: 0 for label in labels_distribution_count.keys()}
        for label, positions in per_file_time_stamps_positions_dict[file]["labels"].items():
            if label not in labels:
                continue
            total = labels_distribution_count[label]
            count_in_train = sum([1 if position < idx else 0 for position in positions])
            count_in_test = sum([1 if position >= idx else 0 for position in positions])
            train_distribution[label] = count_in_train / total 
            test_distribution[label] = count_in_test / total 
            label_weights = get_label_weights(label_time_stamp_count_dict)
            distance = calculate_split_cost(train_distribution, test_distribution, train_ratio=0.33, label_counts=labels_distribution_count, label_weights=label_weights, penalty_weight=2)
            slices.append((idx, distance))
        
    minimal_distance_slice = min(slices, key=lambda x: x[1])


    return minimal_distance_slice[0]


def get_train_test_files_for_label(files, train_ratio=0.33):

    all_labels_in_files = set()
    for file in files:
        all_labels_in_files.update(per_file_time_stamps_positions_dict[file]["labels"].keys())
    
    label_counts_in_files = {label: 0 for label in all_labels_in_files}
    for file in files:
        for label in per_file_time_stamps_positions_dict[file]["labels"].keys():
            label_counts_in_files[label] += len(per_file_time_stamps_positions_dict[file]["labels"][label])

    all_sets = powerset(files)

    backup_splits = []
    splits = []
    for potential_train_files in all_sets:
        potential_test_files = set(files) - set(potential_train_files)
        splits.append((potential_train_files, potential_test_files))

    #get best packup split ratio with mse distance 
    train_split = None
    test_split = None
    
    backup_splits= []
    best_distance = float("inf")
    for potential_train_files, potential_test_files in splits:
        potential_train_distribution = {label: 0 for label in label_time_stamp_count_dict.keys()}
        potential_test_distribution = {label: 0 for label in label_time_stamp_count_dict.keys()}
        for file in potential_test_files:
            for label in per_file_time_stamps_positions_dict[file]["labels"].keys():
                potential_test_distribution[label] += len(per_file_time_stamps_positions_dict[file]["labels"][label])
        for file in potential_train_files:
            for label in per_file_time_stamps_positions_dict[file]["labels"].keys():
                potential_train_distribution[label] += len(per_file_time_stamps_positions_dict[file]["labels"][label])
        for label in label_counts_in_files.keys():
            if label_counts_in_files[label] > 0:
                potential_train_distribution[label] /= label_counts_in_files[label]
                potential_test_distribution[label] /= label_counts_in_files[label]
            else:
                potential_train_distribution[label] = 0
                potential_test_distribution[label] = 0
        label_weights = get_label_weights(label_time_stamp_count_dict)
        distance = calculate_split_cost(potential_train_distribution, potential_test_distribution, train_ratio, label_counts_in_files, label_weights)
        all_train_filled = all(potential_train_distribution[label] >= 1 for label in label_counts_in_files.keys())
        all_test_filled = all(potential_test_distribution[label] >= 1 for label in label_counts_in_files.keys())
        if distance < best_distance and all_train_filled and all_test_filled:
            best_distance = distance
            train_split = potential_train_files
            test_split = potential_test_files
        else: 
            backup_splits.append((potential_train_files, potential_test_files, distance))
    
    if train_split is None or test_split is None:
        min_backup_split = min(backup_splits, key=lambda x: x[2])
        train_split = min_backup_split[0]
        test_split = min_backup_split[1]
        
    return train_split, test_split

In [ ]:
per_label_time_stamps_positions_dict = get_soundscapes_time_stamps_positions()
per_file_time_stamps_positions_dict = transpose_dict_to_file(per_label_time_stamps_positions_dict)
print("Before splitting: distribution", label_time_stamp_count_dict)
print("Before splitting: number of files", len(per_file_time_stamps_positions_dict))        

#### Start with Files that dont contain one file only labels

In [ ]:
#split all files first that dont have file_label_count of 1 for every label
filled_labels = set()
files_2_split = set()
for label, count in sorted(label_time_stamp_count_dict.items(),key=lambda item: item[1]):

    files = list(per_label_time_stamps_positions_dict[label].keys())


    if is_only_one_file(label,label_file_count):
        continue 

    else: 
        splitable_files = [file for file in files if file in per_file_time_stamps_positions_dict.keys()] #check if file stll exist 
        splitable_files = [file for file in splitable_files if not per_file_time_stamps_positions_dict[file]["only_one"]] # check if file is not only one file for its labels
        if len(splitable_files) == 0:
            continue
        else:
            for file in splitable_files:
                files_2_split.add(file)

train_files, test_files = get_train_test_files_for_label(files_2_split)

#update label_time_stamp_count_dict based on the split
for file in train_files:
    file_labels = per_file_time_stamps_positions_dict[file]["labels"].keys()
    for label in file_labels:
        label_time_stamp_count_dict[label] -= len(per_file_time_stamps_positions_dict[file]["labels"][label])
        train_label_time_stamp_count[label] += len(per_file_time_stamps_positions_dict[file]["labels"][label])
        filled_labels.add(label)  # Mark this label as filled
    special_cases_train.append((file,file_labels))
    
    per_file_time_stamps_positions_dict.pop(file)  # Remove the file from consideration in future splits
for file in test_files:
    file_labels = per_file_time_stamps_positions_dict[file]["labels"].keys()
    for label in file_labels:
        label_time_stamp_count_dict[label] -= len(per_file_time_stamps_positions_dict[file]["labels"][label])
        test_label_time_stamp_count[label] += len(per_file_time_stamps_positions_dict[file]["labels"][label])
        filled_labels.add(label)  # Mark this label as filled
    special_cases_test.append((file,file_labels))
    per_file_time_stamps_positions_dict.pop(file)  # Remove the file from consideration in future splits

print("After splitting labels with more than one file: distribution", label_time_stamp_count_dict)
print("Remaining files after splitting labels with more than one file:", len(per_file_time_stamps_positions_dict))       

In [ ]:
not_before_filled = set(label_time_stamp_count_dict.keys()) - filled_labels
print(not_before_filled)
for file, values in per_file_time_stamps_positions_dict.items():
    split_point = find_optimal_time_slice_for_labels(file, not_before_filled)

    train_labels = []
    test_labels = []
    #update label_time_stamp_count_dict based on the split
    for label in values["labels"].keys():
        positions = values["labels"][label]
        count_in_train = sum(1 for position in positions if position < split_point)
        count_in_test = sum(1 for position in positions if position >= split_point)
        train_label_time_stamp_count[label] += count_in_train
        test_label_time_stamp_count[label] += count_in_test
        label_time_stamp_count_dict[label] -= (count_in_train + count_in_test)  # Remove the count for this label as it has been split

        if count_in_train > 0:
            train_labels.append(label)
        if count_in_test > 0:
            test_labels.append(label)
        not_before_filled.discard(label)  # Mark this label as filled
    train_file_name = f".{file.split('.')[-2]}_split_train_{split_point}.ogg"
    print(train_file_name)
    test_file_name = f".{file.split('.')[-2]}_split_test_{split_point}.ogg"
    special_cases_train.append((train_file_name, train_labels))
    special_cases_test.append((test_file_name, test_labels))

#loop over remaining files and split them based on the find_two_third_test_slice_for_every_label function 
print("Train label time stamp count:", train_label_time_stamp_count)
print("Test label time stamp count:", test_label_time_stamp_count)
print("Remaining label time stamp count (should be 0):", label_time_stamp_count_dict)

### Postprocess train file test split

In [ ]:
special_cases_train_df = pd.DataFrame(special_cases_train, columns=["filename", "primary_label"])
special_cases_test_df = pd.DataFrame(special_cases_test, columns=["filename", "primary_label"])

#assertion that all filenames in lower_quantile_df were handled 
lower_quantile_df = get_below_lower_quantile_df(all_files_df)
handled_files = set(special_cases_train_df["filename"].tolist() + special_cases_test_df["filename"].tolist())
print(handled_files)
for filename in lower_quantile_df["filename"]:
    file_handled = any(filename.split(".")[-2].split("/")[-1] in handled_file for handled_file in handled_files)
    if not file_handled:
        raise ValueError(f"File {filename} from lower_quantile_df was not handled in the splitting process. {filename.split(".")[-2].split("/")[-1]}")

#assertion that no new cases popped up 
label_file_count = get_label_file_count()
#update with label counts from special cases
for _, row in special_cases_train_df.iterrows():
    for label in row["primary_label"]:
        label_file_count[label] -= 1
for _, row in special_cases_test_df.iterrows():
    for label in row["primary_label"]:
        label_file_count[label] -= 1

#assertion that no new cases popped up 
new_lower_quantile_df = get_below_lower_quantile_df(all_files_df)
new_lower_quantile_df["below_lower_quantile"] = all_files_df["primary_label"].apply(lambda label_list: any(is_below_lower_quantile(label, label_file_count) for label in label_list))
new_lower_quantile_df =  new_lower_quantile_df[new_lower_quantile_df["below_lower_quantile"] == True]
#remove overlaps with special cases
new_lower_quantile_df = new_lower_quantile_df[~new_lower_quantile_df["filename"].apply(lambda filename: any(filename.split(".")[-2].split("/")[-1] in handled_file for handled_file in handled_files))]

assert new_lower_quantile_df.empty, "There are new files in the lower quantile after the split, some labels might have been missed or new problems might have popped up."

#map back common names to primary labels for the special cases
special_cases_train_df["primary_label"] = special_cases_train_df["primary_label"].apply(lambda label_list: [taxonomy_df[taxonomy_df["common_name"] == label]["primary_label"].values[0] for label in label_list])
special_cases_test_df["primary_label"] = special_cases_test_df["primary_label"].apply(lambda label_list: [taxonomy_df[taxonomy_df["common_name"] == label]["primary_label"].values[0] for label in label_list])


## Multilabel Stratified Split on the rest of the Data

In [ ]:
# perform multilabel split to have every unique label in the train and validation set
train_size = 0.8
test_size = 0.2

In [ ]:
#class mapping 
class_mapping = {label: idx for idx, label in enumerate(sorted(taxonomy_df['primary_label'].unique()))}

In [ ]:
print("class_mapping:", class_mapping)

In [ ]:

random_state = 42
np.random.seed(random_state)

all_files_df["primary_label"] = all_files_df["primary_label"].apply(lambda label_list: [taxonomy_df[taxonomy_df["common_name"] == label]["primary_label"].values[0] for label in label_list])

# Let's target a 70% Train, 15% Val, 15% Test split.
X = np.array([i for i in range(len(all_files_df))])  # Just an index array for splitting
y = np.array([[0] * len(class_mapping) for _ in range(len(all_files_df))])  # one hot vectors of labels for each sample
for idx, labels in enumerate(all_files_df["primary_label"]):
    for label in labels:
        label_idx = class_mapping[label]
        y[idx][label_idx] = 1 

# ---------------------------------------------------------
# Step 1: Split into Temp (70%) and Test (30%)
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)

for temp_index, test_index in msss1.split(X, y):
    X_temp, X_test = X[temp_index], X[test_index]
    y_temp, y_test = y[temp_index], y[test_index]

# ---------------------------------------------------------
# Step 2: Split Temp into Train (70%) and Val (30%)
#recalculate percentages for train and val based on the new temp size
#val_size = val_size / (train_size + val_size)  # Adjust val_size to
#msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=val_size, random_state=random_state)
#
#for train_index, val_index in msss2.split(X_temp, y_temp):
#    X_train, X_val = X_temp[train_index], X_temp[val_index]
#    y_train, y_val = y_temp[train_index], y_temp[val_index]

#skip validation for now 
X_train, y_train = X_temp, y_temp

# ---------------------------------------------------------
# Verification
# ---------------------------------------------------------
print(f"Total samples: {len(X)}")
print(f"Train samples: {len(X_train)}")
#print(f"Val samples:   {len(X_val)}")
print(f"Test samples:  {len(X_test)}\n")

print("Label distributions (sum of positive classes):")
print(f"Original: {y.sum(axis=0)}")
print(f"Train:    {y_train.sum(axis=0)}")
#print(f"Val:      {y_val.sum(axis=0)}")
print(f"Test:     {y_test.sum(axis=0)}")

In [ ]:
#create csvs based on x_train, x_val, x_test
train_df = all_files_df.iloc[X_train]
#val_df = all_files_df.iloc[X_val]
test_df = all_files_df.iloc[X_test]

#fuse with special cases
train_df = pd.concat([train_df, special_cases_train_df], ignore_index=True)
test_df = pd.concat([test_df, special_cases_test_df], ignore_index=True)

train_df.to_csv("./data/train_split.csv", index=False)
#val_df.to_csv(".data/val_split.csv", index=False)
test_df.to_csv("./data/test_split.csv", index=False)

In [ ]:
print(train_df.shape[0], "samples in train set")
#print(val_df.shape[0], "samples in validation set")
print(test_df.shape[0], "samples in test set")

print(train_df.head(5))
#print(val_df.head(5))
print(test_df.head(5))

In [ ]:
def get_label_distribution_count(df):
    #get label distribution 
    df_labels_count = {taxonomy_label: 0 for taxonomy_label in taxonomy_df["primary_label"].unique()}
    for row in df.itertuples():
        labels = row.primary_label

        for label in labels:
            if label in df_labels_count:
                df_labels_count[label] += 1
            else:
                df_labels_count[label] = 1
    return df_labels_count

def plot_label_distribution_count(df_labels_count, title=""):
    print("Number of animals labeled in train soundscapes labels: {}".format(len(df_labels_count)))
    #sort df_labels_count by count
    df_labels_count = dict(sorted(df_labels_count.items(), key=lambda item: item[1], reverse=True))  
    #print("Label distribution in train soundscapes labels:")
    #for label, count in df_labels_count.items():
    #    print(f"{label}: {count}")
    #plot label distribution in train soundscapes labels
    plt.figure(figsize=(10,5))
    plt.bar(df_labels_count.keys(), df_labels_count.values())
    plt.xlabel("Label")
    plt.ylabel("Count")
    plt.title(title +"Label Distribution")
    plt.xticks(rotation=45)

In [ ]:
#print label mismatches 
train_labels = set([label for labels in train_df["primary_label"] for label in labels])
test_labels = set([label for labels in test_df["primary_label"] for label in labels])
print("Labels in train set but not in test set:", train_labels - test_labels)

In [ ]:
for df, name in zip([train_df, test_df], ["Train", "Test"]):
    print(f"\n{name} Set:")
    df_labels_count = get_label_distribution_count(df)
    plot_label_distribution_count(df_labels_count, title=f"{name} Set Label Distribution")